# A 90% Interval Is Not a Promise

Executable companion to the [conformal prediction article](article.md). The population, response, conditional noise, covariate shift, and synthetic oracle ratios are all explicitly constructed. The eight publication figures and ten CSV tables are generated by [analysis.py](analysis.py), using independent seeded streams. This notebook runs a reduced-size demonstration.

## 1. The finite-sample rank

Given nonnegative held-out calibration scores, the split-conformal threshold uses rank k = ceil((n + 1)(1 − alpha)), or infinity if k exceeds n. A default interpolated percentile is not a substitute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import ALPHA, METHODS, run_study, split_threshold

assert split_threshold(np.arange(10.0), 0.1) == 9.0
assert np.isinf(split_threshold(np.arange(5.0), 0.01))
study = run_study(quick=True)
print('Finite calibration group B count:', study.summary['cal_group_b'])
print('Global conformal halfwidth:', round(study.calibration.global_q, 4))
print('Across-split mean coverage:', round(study.summary['replicate_global_coverage_mean'], 4))

## 2. Marginal validity is not group coverage

Even with an accurate mean model, heteroscedasticity makes one global score threshold too permissive for group A and too narrow for group B. Mondrian group calibration targets each discrete group separately, given within-group exchangeability. Training-fitted normalized scores target widths adaptively but carry only pooled marginal validity by default.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for j, method in enumerate(METHODS):
    values = [study.coverage['baseline'][method][key] for key in ('marginal', 'group_a', 'group_b')]
    ax.bar(np.arange(3) + (j - 1) * 0.25, values, 0.23, label=method)
ax.axhline(1 - ALPHA, color='black', linestyle='--', label='Nominal 90%')
ax.set(xticks=range(3), xticklabels=['Pooled', 'Group A', 'Group B'], ylabel='Empirical coverage', ylim=(0, 1.02))
ax.legend()
plt.show()

## 3. Covariate shift and conditional shift are different failures

The first deployment population raises group B prevalence from 20% to 60% while holding the conditional response distribution fixed. The weighted method uses the exactly known synthetic density ratios, which would not generally be available in a real deployment. The second population retains group frequencies but multiplies group B response-noise SD by 1.65: group-conditional validity no longer applies.

In [ ]:
for scenario in ('baseline', 'mixture', 'conditional'):
    print('\n', scenario)
    for method, metrics in study.coverage[scenario].items():
        print(method, 'pooled:', round(metrics['marginal'], 3), 'group B:', round(metrics['group_b'], 3))
assert study.coverage['mixture']['Global']['marginal'] < study.coverage['baseline']['Global']['marginal']
assert study.coverage['conditional']['Mondrian']['group_b'] < study.coverage['baseline']['Mondrian']['group_b']

## 4. Temporal scores break the original exchangeability argument

The stream is AR(1), with a variance increase at time 600. The rolling threshold is trained on the previous 120 scores and updated only after observing the latest outcome. This is a heuristic response to drift, not a proof of online or conditional coverage.

In [ ]:
stream = study.stream
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(stream['t'], stream['fixed_trailing'], label='Static, trailing 100')
ax.plot(stream['t'], stream['rolling_trailing'], label='Rolling, trailing 100')
ax.axhline(0.9, color='black', linestyle=':')
ax.axvline(600, color='black', linestyle='--')
ax.set(xlabel='Time', ylabel='Realized trailing coverage')
ax.legend()
plt.show()

## 5. Simultaneous prediction for three targets

Separate marginal 90% statements do not provide 90% probability of covering all three outputs at once. Bonferroni applies alpha/3 to each response to control the familywise miss event without relying on independence across targets. One realized test fraction can still fall just below nominal, even when a marginal-over-calibration theorem holds.

In [ ]:
joint = study.multitarget
print('Ordinary joint:', round(joint['plain_joint'], 3))
print('Bonferroni joint:', round(joint['bonf_joint'], 3))
print('Full-width totals:', round(joint['plain_total_width'], 2), round(joint['bonf_total_width'], 2))
assert joint['bonf_joint'] >= joint['plain_joint']